# BioAgent-Lite: A Resource-Efficient Multi-Agent Framework for Reliable Gene-Expression Analysis
**Course:** CSE 763 Advanced Bioinformatics / CSE 443 Bioinformatics  
**Project:** Machine Learning & Agentic Bioinformatics Term Project  
**Target Hardware:** Consumer Windows PC (NVIDIA RTX 5050 8 GB VRAM, 16 GB RAM)  
**Execution Runtime:** Local GGUF via `llama.cpp` CUDA 12.4 server (Zero cloud API reliance)  

---
## Executive Summary & Research Motivation
Large language models (LLMs) suffer from severe hallucinations when applied to bioinformatics: they invent statistical thresholds, confuse adjusted p-values ($padj$) with raw p-values, fabricate genes, and over-interpret biological mechanisms.

**BioAgent-Lite** solves this by establishing the **Deterministic Sandboxing Principle**:
1. Numerical calculation is strictly delegated to deterministic software (`pydeseq2`, `gseapy`).
2. The language model is restricted to planning, biological interpretation with mandatory scientific hedging, explicit evidence grounding, and adversarial verification (Critic).
3. The entire pipeline executes locally on consumer hardware within a **4,919 MB VRAM footprint** at **38.5 tokens/second**.

This notebook provides the complete, self-contained demonstration of data ingestion, reference computation, multi-agent execution, baselines, confusion matrices, ROC-AUC curves, and robustness evaluations.

---
## 1. Dataset Suite Ingestion & Quality Control
We evaluate BioAgent-Lite across four benchmark transcriptomic datasets:
1. **GSE202768:** Primary development dataset (16 samples, DOX-inducible PRRX1 knockdown in Hs578 TNBC cell lines, raw counts).
2. **GSE113650:** External validation dataset (6 samples, TNBC vs non-TNBC cell line CuffDiff output).
3. **GSE95304:** Held-out perturbation dataset (20 samples, shMyc in breast cancer cell lines, FPKM tracking).
4. **GSE336903:** Smoke test dataset (6 samples, mouse model *Mus musculus*).

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Load and display Table 1: Benchmark Dataset Summary
df_datasets = pd.read_csv('outputs/tables/dataset_summary.csv')
display(df_datasets)

---
## 2. Deterministic Reference Analysis (`pydeseq2` & `gseapy`)
The statistical backbone executes:
- **Wald negative-binomial tests** with Cook's distance filtering and Benjamini-Hochberg FDR ($padj < 0.05$).
- **Over-representation analysis** across Gene Ontology (Biological Process), Reactome, and KEGG pathways.
- **Scientific Data-Type Refusal Guard (§13):** Dynamic inspection refuses negative-binomial modeling on normalized FPKM floats, routing to descriptive fold-changes.

In [ ]:
print("Displaying Differential Expression Volcano Plot (GSE202768, PRRX1 KD)...")
display(Image(filename='outputs/figures/volcano_plot_GSE202768.png'))

print("Displaying Top Reactome Pathways (GSE202768)...")
display(Image(filename='outputs/figures/pathways_GSE202768.png'))

---
## 3. The 5-Agent LangGraph Workflow & Adversarial Error Correction
The system couples five specialized roles into a cyclic directed graph:
1. **Planner:** Formulates structured JSON analysis contracts.
2. **Computational Analyst:** Deterministic dispatcher (`pydeseq2` / `gseapy`).
3. **Biological Interpreter:** Categorizes claims into `observed`, `supported`, and `hypothesis` with mandatory hedging.
4. **Evidence Retriever:** Interrogates computed tables to assign verifiable lineage records.
5. **Critic / Verifier:** 4-layer audit (statistical, direction concordance, pathway existence, hedging) with a bounded self-correction loop.

### Empirical Case Study (RQ4 / H4):
In run `dev_20260905T165240Z`, the Interpreter claimed gene **$CD82$** was down-regulated. The Critic cross-checked the DESeq2 output, observed that $\log_2FC = +1.14$ (positive fold change), flagged a **Direction Inversion Error (`FAIL`)**, and triggered the correction loop. In Iteration 1, the erroneous claim was corrected, and the Critic issued a `PASS`.

In [ ]:
# Inspect sample agent trajectory trace
import json
with open('outputs/manifest.json', 'r') as f:
    manifest = json.load(f)
print(f"Active Experiment: {manifest['experiment']}")
print(f"Dataset: {manifest['dataset']} ({manifest['contrast']})")
print(f"Genes tested: {manifest['genes_tested']}, Significant (padj < 0.05): {manifest['n_sig_padj0.05']}")
print(f"PRRX1 knockdown verification: {manifest['prrx1_check']}")

---
## 4. Baseline Architecture Benchmarks (RQ1 & RQ2)
We evaluate:
- **Baseline 1 (LLM-Only):** Direct prompt reasoning without tool access.
- **Baseline 2 (Single Tool-Agent):** Standard ReAct-style agent with tool execution.
- **BioAgent-Lite (Proposed):** Full 5-role architecture with explicit Critic verification.

In [ ]:
df_arch = pd.read_csv('outputs/tables/architecture_comparison.csv')
display(df_arch)

display(Image(filename='outputs/figures/comprehensive_metrics_comparison.png'))

---
## 5. Machine Learning Evaluation: Confusion Matrices & ROC-AUC Curves
Treating significant differential gene identification ($padj < 0.05, |\log_2FC| \ge 1.0$) as a binary classification problem:
- **True Positives (TP):** Correctly identified significant differential genes.
- **False Positives (FP):** Hallucinated genes or non-significant genes called positive.
- **False Negatives (FN):** True significant genes missed.
- **True Negatives (TN):** Correctly rejected non-differential genes.

In [ ]:
print("Confusion Matrices across Architectures:")
display(Image(filename='outputs/figures/confusion_matrices.png'))

print("Receiver Operating Characteristic (ROC) & Precision-Recall (PR) Curves:")
display(Image(filename='outputs/figures/roc_auc_curves.png'))

### Key Findings from ML Curves:
1. **BioAgent-Lite achieves an AUC of 1.000 and AP of 1.000**, with 0 false positives (100% precision) and 0 false negatives (100% recall).
2. **Single Tool-Agent achieves an AUC of 0.800**, suffering from context compression which caused 8 false negatives (Recall = 60.0%).
3. **LLM-Only collapses to an AUC of 0.520** (barely above random guessing), dominated by a 140% hallucination rate.

---
## 6. Ablation Studies & Adversarial Robustness Suite (RQ3 & RQ4)
To verify role necessity, we tested:
- **Ablation A1:** Remove Planner.
- **Ablation A3:** Remove Evidence Retriever.
- **Ablation A4:** Remove Critic.

In [ ]:
df_ablation = pd.read_csv('outputs/tables/ablation_results.csv')
display(df_ablation)

display(Image(filename='outputs/figures/ablation_impact_breakdown.png'))

print("\nAdversarial Robustness Stress Tests (R1 to R4):")
df_robust = pd.read_csv('outputs/tables/robustness_results.csv')
display(df_robust)

---
## 7. Model Comparison & Local Hardware Resource Footprint (RQ5)
Evaluating local model execution on consumer hardware (NVIDIA RTX 5050 8 GB VRAM):

In [ ]:
df_models = pd.read_csv('outputs/tables/model_comparison.csv')
display(df_models)

df_resources = pd.read_csv('outputs/tables/resource_usage.csv')
display(df_resources)

---
## 8. Conclusion
BioAgent-Lite demonstrates that **small 4B parameter language models (Google Gemma 3 4B Q8) running locally on consumer hardware can match or exceed cloud-scale agent reliability** when paired with:
1. Deterministic bioinformatics sandboxing (`pydeseq2`, `gseapy`).
2. Pydantic typed claim taxonomy and scientific hedging.
3. Explicit evidence lineage grounding.
4. An adversarial Critic with bounded self-correction.

All code, data, figures, and benchmark tables are 100% reproducible offline.